# Day 2 Optional Exercises --- Solutions



## pandas Exercises

Since practice is key to learning how to work with data in Python, it might be useful to know where to get datasets to practice with. The main way I (GK) have done this is to "steal" them from **R**!

**R** datasets are great--they are often clean, well-documented, and pedagogically invaluable. You also do not need to know the first thing about **R** to use them.

[Here is a list of some of the datasets included in the **datasets** package (R Core Team, 2026) in **R**](https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/00Index.html)

We will now install the **statsmodels** (Seabold & Perktold, 2010) Python package to get an **R** dataset into Python. More on this package tomorrow (day 3)!

(Conda break)

In the below code, we will load the `Titanicp` dataset ('p' for 'passengers'; Dawson, 1995; Harrell, 2002) from the **R** package **vcdExtra** (Friendly & Klorfine, 2026). The dataset is the first argument, and the second argument is the **R** package it is from.

In [72]:
import statsmodels.api as sm

titanicp = sm.datasets.get_rdataset("Titanicp", "vcdExtra").data

# Note: I (GK) always forget to use .data at the end here. If you are
# like me and forget, you will get a "dataset object" instead of a DataFrame.

# Can use .title or .__doc__ instead of .data to get a description of the dataset

titanicp.head()

,pclass,survived,sex,age,sibsp,parch
0,1st,survived,female,29.0000,0,0
1,1st,survived,male,0.9167,1,2
2,1st,died,female,2.0000,1,2
3,1st,died,male,30.0000,1,2
4,1st,died,female,25.0000,1,2


In the above dataset, each row represents a single passenger aboard the Titanic.

The columns (variables) are:

- `pclass`: passenger class (1st, 2nd, 3rd)
- `survived`: whether the passenger survived (`"survived"`) or not (`"died"`)
- `sex` of the passenger
- `age` of the passenger in years
- `sibsp`: number of siblings or spouses aboard
- `parch`: number of parents or children aboard

There are some missing values in this data. For simplicity, we will drop any rows containing them. This is done via the **pandas** (The pandas development team, 2026) `DataFrame.dropna()` method:

In [73]:
titanicp = titanicp.dropna()

Now that we have our data, let's explore it.

First, take the mean of the `age` variable:

In [74]:
titanicp["age"].mean()

np.float64(29.881134512434034)

Now find the mean age of passengers who `survived` and those who `died`:

Hint: use `.groupby()`

In [75]:
titanicp.groupby("survived").describe()

# or

titanicp.groupby("survived")["age"].mean()

survived
died        30.545369
survived    28.918228
Name: age, dtype: float64

Use `.loc` to get the difference between these two means:

Note: depending on your answer to the above question, you may need to use a tuple for the column entry in `.loc`. The first element of the tuple will be the "umbrella" heading, and the second element will be the subheading.

In [76]:
tp_grouped = titanicp.groupby("survived")["age"].mean()

tp_grouped.loc["died"] - tp_grouped.loc["survived"]

# or

tp_desc = titanicp.groupby("survived").describe()

tp_desc.loc["died", ("age", "mean")] - tp_desc.loc["survived", ("age", "mean")]

np.float64(1.6271407175962231)

Let's make a new column that converts `survived` to 1 and `died` to 0.

As a first step, write a function that takes variable `x` and returns 1 if `x` is `"survived"` and 0 if `x` is `"died"`. Name this function `binary_survival()`:

In [77]:
def binary_survival(x):
    if x == "survived":
        return 1
    elif x == "died":
        return 0

Just as **pandas** has built-in functions such as the `.mean()` method, it also allows us to use custom functions, like the one we just created. This is done through the `.apply()` method, where ones function name is supplied within the parentheses.

Select the `survived` column and then apply our `binary_survival` function to it:

In [78]:
titanicp["survived"].apply(binary_survival)

0       1
1       1
2       0
3       0
4       0
       ..
1301    0
1304    0
1306    0
1307    0
1308    0
Name: survived, Length: 1046, dtype: int64

Save the result of the above code as a new column named `survived_binary`:

In [79]:
titanicp["survived_binary"] = titanicp["survived"].apply(binary_survival)

titanicp.head()

,pclass,survived,sex,age,sibsp,parch,survived_binary
0,1st,survived,female,29.0000,0,0,1
1,1st,survived,male,0.9167,1,2,1
2,1st,died,female,2.0000,1,2,0
3,1st,died,male,30.0000,1,2,0
4,1st,died,female,25.0000,1,2,0


The mean of a binary variable is useful--it gives us the proportion of 1s. In this case, this is the proportion of passengers that survived.

Not important, but here is the math behind this if it helps:


\begin{align}
\hat{\mu} &= \frac{1}{n}\sum_{i=1}^{n} x_i \\

\hat{\mu} &= \frac{1}{n} (0 + 0 + ... + 0 + 1 + 1 + ... + 1) \\

\hat{\mu} &= \frac{1}{n} (1 + 1 + ... + 1) \\ \\

\hat{\mu} &= \frac{1 + 1 + ... + 1}{n} \\
\end{align}

Get the proportion survived for each `pclass`. What do you notice?


In [80]:
titanicp.groupby("pclass")["survived_binary"].mean()

# Rich were more likely to live (on average)!

pclass
1st    0.637324
2nd    0.440613
3rd    0.261477
Name: survived_binary, dtype: float64

Add a second grouping variable of `sex`. What do you notice?

In [81]:
titanicp.groupby(["pclass", "sex"])["survived_binary"].mean()

pclass  sex   
1st     female    0.962406
        male      0.350993
2nd     female    0.893204
        male      0.145570
3rd     female    0.473684
        male      0.169054
Name: survived_binary, dtype: float64

Use the operations you have learned today to explore the association between `age` and `pclass`. What did you find?

In [82]:
titanicp.groupby("pclass")["age"].describe()

,count,mean,std,min,25%,50%,75%,max
pclass,,,,,,,,
1st,284.0,39.159918,14.548059,0.9167,28.0,39.0,50.0,80.0
2nd,261.0,29.506705,13.638628,0.6667,22.0,29.0,36.0,70.0
3rd,501.0,24.816367,11.958202,0.1667,18.0,24.0,32.0,74.0


How many infants (i.e., passengers less than one year old) were aboard the ship?

In [83]:
len(titanicp[titanicp["age"] < 1]) # 12 infants

# or

titanicp[titanicp["age"] < 1].shape

# or

titanicp[titanicp["age"] < 1].info()

<class 'pandas.DataFrame'>
Index: 12 entries, 1 to 1240
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   pclass           12 non-null     str    
 1   survived         12 non-null     str    
 2   sex              12 non-null     str    
 3   age              12 non-null     float64
 4   sibsp            12 non-null     int64  
 5   parch            12 non-null     int64  
 6   survived_binary  12 non-null     int64  
dtypes: float64(1), int64(3), str(3)
memory usage: 768.0 bytes


How many infants of first class were on the ship?

In [84]:
len(titanicp[(titanicp["age"] < 1) & (titanicp["pclass"] == "1st")])

1

What were the classes of the other infants?

Hint: use `.apply(len)` after grouping and indexing

In [ ]:
titanicp[(titanicp["age"] < 1)].groupby("pclass")["pclass"].apply(len)

# or

# Fancier(?) way
titanicp[(titanicp["age"] < 1) & (titanicp["pclass"] != "1st")] # Removed len() from last question's answer and switched == to !=

titanicp[(titanicp["age"] < 1) & (titanicp["pclass"] != "1st")].groupby("pclass")["pclass"].apply(len)

pclass
2nd    4
3rd    7
Name: pclass, dtype: int64

## References

 Dawson, Robert J. MacG. (1995), The ‘Unusual Episode’ Data Revisited. Journal of Statistics Education, 3. doi:10.1080/10691898.1995.11910499.
 
 Friendly M, Klorfine G (2026). _vcdExtra: 'vcd' Extensions and Additions_. R package version 0.9.5, <[https://friendly.github.io/vcdExtra/](https://friendly.github.io/vcdExtra/)>.

Harrell, F.E. Jr. (2002). Titanic Data. [https://hbiostat.org/data/repo/titanic](https://hbiostat.org/data/repo/titanic)

 R Core Team (2026). _R: A Language and Environment for Statistical Computing_. R Foundation for Statistical Computing, Vienna, Austria. [https://www.R-project.org/](https://www.R-project.org/)

 Seabold, Skipper, and Josef Perktold. “statsmodels: Econometric and statistical modeling with python.” Proceedings of the 9th Python in Science Conference. 2010. [http://conference.scipy.org/proceedings/scipy2010/pdfs/seabold.pdf](http://conference.scipy.org/proceedings/scipy2010/pdfs/seabold.pdf)

 The pandas development team. (2026). pandas-dev/pandas: Pandas (v3.0.3). Zenodo. [https://doi.org/10.5281/zenodo.20127038](https://doi.org/10.5281/zenodo.20127038)
